In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv("train.csv")
df = df.drop_duplicates()
df.shape

(159571, 8)

In [3]:
df.columns = df.columns.str.lower().str.strip().str.replace(" ", "_")
df = df.drop(columns="id")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 159571 entries, 0 to 159570
Data columns (total 7 columns):
 #   Column         Non-Null Count   Dtype 
---  ------         --------------   ----- 
 0   comment_text   159571 non-null  object
 1   toxic          159571 non-null  int64 
 2   severe_toxic   159571 non-null  int64 
 3   obscene        159571 non-null  int64 
 4   threat         159571 non-null  int64 
 5   insult         159571 non-null  int64 
 6   identity_hate  159571 non-null  int64 
dtypes: int64(6), object(1)
memory usage: 8.5+ MB


In [4]:
df.isna().sum()

comment_text     0
toxic            0
severe_toxic     0
obscene          0
threat           0
insult           0
identity_hate    0
dtype: int64

In [5]:
for col in df.columns:
    print(col, (df[col] == " ").sum())

comment_text 0
toxic 0
severe_toxic 0
obscene 0
threat 0
insult 0
identity_hate 0


In [6]:
df["comment_length"] = df["comment_text"].str.len()
df["word_count"] = df["comment_text"].str.split().str.len()
df.head()

,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate,comment_length,word_count
0,Explanation\nWhy the edits made under my usern...,0,0,0,0,0,0,264,43
1,D'aww! He matches this background colour I'm s...,0,0,0,0,0,0,112,17
2,"Hey man, I'm really not trying to edit war. It...",0,0,0,0,0,0,233,42
3,"""\nMore\nI can't make any real suggestions on ...",0,0,0,0,0,0,622,113
4,"You, sir, are my hero. Any chance you remember...",0,0,0,0,0,0,67,13


In [7]:
import re 
import string
import contractions
import nltk
import emoji

from nltk.corpus import stopwords, wordnet
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")
nltk.download("averaged_perceptron_tagger")
nltk.download("averaged_perceptron_tagger_eng")

stop_words = set(stopwords.words("english"))
correct_negations = {
    "not",
    "cannot",
    "no",
    "nor",
    "never",
    "none",
    "nothing",
    "nobody",
    "nowhere",
    "neither"
}
custom_stopwords = stop_words - correct_negations

lemmatizer = WordNetLemmatizer()

def get_wordnet_pos(word):
    tag = pos_tag([word])[0][1]

    if tag.startswith("J"):
        return wordnet.ADJ
    elif tag.startswith("V"):
        return wordnet.VERB
    elif tag.startswith("N"):
        return wordnet.NOUN
    elif tag.startswith("R"):
        return wordnet.ADV
    else:
        return wordnet.NOUN

def clean_text(txt):
    txt = txt.lower()
    txt = re.sub(r"https?://\S+|www\.\S+", "", txt)
    txt = re.sub(r"<[^>]+>", "", txt)
    txt = emoji.replace_emoji(txt, replace = "")
    txt = contractions.fix(txt)
    txt = txt.translate(str.maketrans("","", string.punctuation))
    tokens = word_tokenize(txt)
    tokens = [word for word in tokens if word not in custom_stopwords]
    tokens = [lemmatizer.lemmatize(word, get_wordnet_pos(word)) for word in tokens]
    tokens = [word for word in tokens if word.strip()]
    return " ".join(tokens)
    
    

# df["comment_text"].apply(clean_text)
    

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\harsh\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\harsh\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\harsh\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\harsh\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\harsh\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\harsh\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nl

In [8]:
from nltk import pos_tag
pos_tag(["study", "eating"])

[('study', 'NN'), ('eating', 'VBG')]

In [9]:
clean_text("i will KILL you!!!")
df["clean_comment"] = df["comment_text"].apply(clean_text)

In [10]:
from sklearn.model_selection import train_test_split

X = df[['comment_length', 'word_count', 'clean_comment']]
y = df[['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=.2, random_state=42
)

In [11]:
X_train

,comment_length,word_count,clean_comment
140030,111,19,grandma terri burn trash grandma terri trash h...
159124,502,91,9 may 2009 utc would easy admit member involve...
60006,4573,722,objectivity discussion doubtful nonexistent 1 ...
65432,36,8,shelly shock shelly shock
154979,243,45,not care refer ong teng cheong talk page la go...
...,...,...,...
119879,51,5,redirect talkjohn loveday experimental physicist
103694,50,10,back post line reference
131932,346,60,not stop sometimes germanic equal germany toda...
146867,175,23,british band think mistaken scottish gaelic wo...


In [12]:
y_train

,toxic,severe_toxic,obscene,threat,insult,identity_hate
140030,1,0,0,0,0,0
159124,0,0,0,0,0,0
60006,0,0,0,0,0,0
65432,0,0,0,0,0,0
154979,0,0,0,0,0,0
...,...,...,...,...,...,...
119879,0,0,0,0,0,0
103694,0,0,0,0,0,0
131932,1,0,0,0,0,0
146867,0,0,0,0,0,0


In [13]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import MaxAbsScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import StandardScaler

text_pipe = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=150_000))
])

num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

lr_preprocessor = ColumnTransformer(
    transformers=[
        ("text_pipe", text_pipe, "clean_comment"),
        ("num_pipe", num_pipeline, ["comment_length", "word_count"])
    ]
)

In [14]:
lr_pipe = Pipeline([
    ("preprocessor", lr_preprocessor),
    ("lr", OneVsRestClassifier(
        LogisticRegression(max_iter = 1000, random_state=42, class_weight='balanced'),
        verbose = 3,
        n_jobs = -1
    ))
])

lr_pipe.fit(X_train, y_train)

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done   4 out of   6 | elapsed:   14.6s remaining:    7.3s
[Parallel(n_jobs=-1)]: Done   6 out of   6 | elapsed:   15.8s finished


,steps,"[('preprocessor', ...), ('lr', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('text_pipe', ...), ('num_pipe', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [15]:
svc_preprocessor = ColumnTransformer(
    transformers=[
        ("text_pipe", text_pipe, "clean_comment"),
        ("num_pipe", num_pipeline, ["comment_length", "word_count"])
    ]
)

svc_pipe = Pipeline([
    ("preprocessor", svc_preprocessor),
    ("linear_svc", OneVsRestClassifier(
        LinearSVC(random_state=42, class_weight='balanced'),
        verbose = 3,
        n_jobs = -1
    ))
])

svc_pipe.fit(X_train, y_train)

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done   4 out of   6 | elapsed:   24.7s remaining:   12.3s
[Parallel(n_jobs=-1)]: Done   6 out of   6 | elapsed:   27.7s finished


,steps,"[('preprocessor', ...), ('linear_svc', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('text_pipe', ...), ('num_pipe', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [16]:
nb_preprocessor = ColumnTransformer(
    transformers=[
        ("text_pipe", text_pipe, "clean_comment"),
        ("num_imputer", SimpleImputer(strategy="median"), ["comment_length", "word_count"])
    ]
)

nb_pipe = Pipeline([
    ("preprocessor", nb_preprocessor),
    ("nb", OneVsRestClassifier(
        MultinomialNB(),
        verbose= 3,
        n_jobs= -1
    ))
])

nb_pipe.fit(X_train, y_train)

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done   4 out of   6 | elapsed:    0.1s remaining:    0.0s
[Parallel(n_jobs=-1)]: Done   6 out of   6 | elapsed:    0.1s finished


,steps,"[('preprocessor', ...), ('nb', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('text_pipe', ...), ('num_imputer', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [17]:
from sklearn.metrics import f1_score, hamming_loss

lr_pred = lr_pipe.predict(X_test)
nb_pred = nb_pipe.predict(X_test)
svc_pred = svc_pipe.predict(X_test)

print("Logistic Regression")
print("Macro F1:", f1_score(y_test, lr_pred, average="macro"))
print("Micro F1:", f1_score(y_test, lr_pred, average="micro"))
print("Hamming Loss:", hamming_loss(y_test, lr_pred))

print("\nNaive Bayes")
print("Macro F1:", f1_score(y_test, nb_pred, average="macro"))
print("Micro F1:", f1_score(y_test, nb_pred, average="micro"))
print("Hamming Loss:", hamming_loss(y_test, nb_pred))

print("\nLinear SVC")
print("Macro F1:", f1_score(y_test, svc_pred, average="macro"))
print("Micro F1:", f1_score(y_test, svc_pred, average="micro"))
print("Hamming Loss:", hamming_loss(y_test, svc_pred))

Logistic Regression
Macro F1: 0.5394933590245132
Micro F1: 0.6719345747914018
Hamming Loss: 0.031004229985900047

Naive Bayes
Macro F1: 0.10782905250533281
Micro F1: 0.22830822855720154
Hamming Loss: 0.03237244764739673

Linear SVC
Macro F1: 0.5664980129500063
Micro F1: 0.6988731868268692
Hamming Loss: 0.025259804689539925


In [18]:
from sklearn.model_selection import GridSearchCV

lr_param_grid = {
    # TF-IDF parameters
    "preprocessor__text_pipe__tfidf__ngram_range": [(1, 1), (1, 2)],
    "preprocessor__text_pipe__tfidf__min_df": [1, 2, 5],
    "preprocessor__text_pipe__tfidf__max_df": [0.95, 1.0],

    # Logistic Regression parameters
    "lr__estimator__C": [0.1, 1, 5],
    "lr__estimator__class_weight": [None, "balanced"]
}

grid_lr = GridSearchCV(
    lr_pipe,
    lr_param_grid,
    cv = 3, 
    verbose = 3,
    n_jobs = -1,
    scoring = "f1_macro"
)

grid_lr.fit(X_train, y_train)

Fitting 3 folds for each of 72 candidates, totalling 216 fits


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done   4 out of   6 | elapsed:   17.1s remaining:    8.5s
[Parallel(n_jobs=-1)]: Done   6 out of   6 | elapsed:   18.8s finished


,estimator,Pipeline(step... verbose=3))])
,param_grid,"{'lr__estimator__C': [0.1, 1, ...], 'lr__estimator__class_weight': [None, 'balanced'], 'preprocessor__text_pipe__tfidf__max_df': [0.95, 1.0], 'preprocessor__text_pipe__tfidf__min_df': [1, 2, ...], ...}"
,scoring,'f1_macro'
,n_jobs,-1
,refit,True
,cv,3
,verbose,3
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('text_pipe', ...), ('num_pipe', ...)]"


In [19]:
svc_param_grid = {
    "preprocessor__text_pipe__tfidf__ngram_range": [(1, 1), (1, 2)],
    "preprocessor__text_pipe__tfidf__min_df": [1, 5],
    "preprocessor__text_pipe__tfidf__max_df": [0.95, 1.0],

    "linear_svc__estimator__C": [0.1, 1, 5],
    "linear_svc__estimator__class_weight": [None, "balanced"]
}

grid_svc = GridSearchCV(
    svc_pipe,
    svc_param_grid,
    cv = 3, 
    verbose = 3,
    n_jobs = -1,
    scoring = "f1_macro"
)

grid_svc.fit(X_train, y_train)

Fitting 3 folds for each of 48 candidates, totalling 144 fits


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done   4 out of   6 | elapsed:   29.5s remaining:   14.7s
[Parallel(n_jobs=-1)]: Done   6 out of   6 | elapsed:   32.5s finished


,estimator,Pipeline(step... verbose=3))])
,param_grid,"{'linear_svc__estimator__C': [0.1, 1, ...], 'linear_svc__estimator__class_weight': [None, 'balanced'], 'preprocessor__text_pipe__tfidf__max_df': [0.95, 1.0], 'preprocessor__text_pipe__tfidf__min_df': [1, 5], ...}"
,scoring,'f1_macro'
,n_jobs,-1
,refit,True
,cv,3
,verbose,3
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('text_pipe', ...), ('num_pipe', ...)]"


In [20]:
from sklearn.metrics import f1_score, hamming_loss

lr_pred = grid_lr.predict(X_test)
svc_pred = grid_svc.predict(X_test)

print("Logistic Regression")
print("Macro F1:", f1_score(y_test, lr_pred, average="macro"))
print("Micro F1:", f1_score(y_test, lr_pred, average="micro"))
print("Hamming Loss:", hamming_loss(y_test, lr_pred))

print("\nLinear SVC")
print("Macro F1:", f1_score(y_test, svc_pred, average="macro"))
print("Micro F1:", f1_score(y_test, svc_pred, average="micro"))
print("Hamming Loss:", hamming_loss(y_test, svc_pred))

Logistic Regression
Macro F1: 0.5850250212252139
Micro F1: 0.7119935068989199
Hamming Loss: 0.024090030811008408

Linear SVC
Macro F1: 0.5931858877104407
Micro F1: 0.7151290044438549
Hamming Loss: 0.022429369679878845


In [49]:
from sklearn.model_selection import train_test_split

X_train_sub, X_val, y_train_sub, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.2,
    random_state=42
)

In [50]:
import copy
best_svc = copy.deepcopy(grid_svc.best_estimator_)

best_svc.fit(X_train_sub, y_train_sub)

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done   4 out of   6 | elapsed:   24.2s remaining:   12.1s
[Parallel(n_jobs=-1)]: Done   6 out of   6 | elapsed:   26.3s finished


,steps,"[('preprocessor', ...), ('linear_svc', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('text_pipe', ...), ('num_pipe', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [64]:
val_scores = best_svc.decision_function(X_val)
# val_scores

array([[-1.73102592, -1.60358688, -1.63482491, -1.4397905 , -1.37842651,
        -1.79022692],
       [-1.16811973, -1.31858109, -1.50323871, -1.67928182, -1.66351629,
        -1.46252955],
       [-1.67523104, -1.48937746, -1.40771991, -1.50272478, -1.48743043,
        -1.3928256 ],
       ...,
       [-1.17023968, -1.95402218, -1.28539548, -1.50336217, -1.01549817,
        -1.83227655],
       [-2.06619807, -1.77409028, -1.58536522, -1.75291184, -1.99604965,
        -1.89598401],
       [-1.80215583, -1.5046956 , -1.26390109, -1.61561144, -1.34188945,
        -1.19677245]], shape=(25532, 6))

In [52]:
import numpy as np
from sklearn.metrics import f1_score

thresholds = np.arange(-0.5, 0.91, 0.05)

best_threshold = 0
best_macro_f1 = 0

for threshold in thresholds:

    val_pred = (val_scores >= threshold).astype(int)

    macro_f1 = f1_score(
        y_val,
        val_pred,
        average="macro"
    )

    if macro_f1 > best_macro_f1:
        best_macro_f1 = macro_f1
        best_threshold = threshold

print("Best Global Threshold:", best_threshold)
print("Best Validation Macro F1:", best_macro_f1)

Best Global Threshold: 0.04999999999999982
Best Validation Macro F1: 0.588886064905448


In [53]:
test_scores = best_svc.decision_function(X_test)

In [54]:
test_pred_threshold = (
    test_scores >= best_threshold
).astype(int)

In [55]:
from sklearn.metrics import f1_score, hamming_loss

print("Linear SVC - Global Threshold Tuned")

print("Macro F1:",
      f1_score(
          y_test,
          test_pred_threshold,
          average="macro"
      ))

print("Micro F1:",
      f1_score(
          y_test,
          test_pred_threshold,
          average="micro"
      ))

print("Hamming Loss:",
      hamming_loss(
          y_test,
          test_pred_threshold
      ))

Linear SVC - Global Threshold Tuned
Macro F1: 0.5937628164979291
Micro F1: 0.7167033571821684
Hamming Loss: 0.021505039427646352


In [56]:
import numpy as np
from sklearn.metrics import f1_score

# Use your best tuned SVC
best_svc = copy.deepcopy(grid_svc.best_estimator_)

# Train on training subset
X_train_sub, X_val, y_train_sub, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.2,
    random_state=42
)

best_svc.fit(X_train_sub, y_train_sub)

# Decision scores on validation set
val_scores = best_svc.decision_function(X_val)

# Store best threshold for each label
best_thresholds_svc = []

for i, label in enumerate(y_train.columns):

    best_threshold = 0
    best_f1 = 0

    for threshold in np.arange(-0.5, 0.91, 0.05):

        val_pred = (val_scores[:, i] >= threshold).astype(int)

        f1 = f1_score(
            y_val.iloc[:, i],
            val_pred
        )

        if f1 > best_f1:
            best_f1 = f1
            best_threshold = threshold

    best_thresholds_svc.append(best_threshold)

    print(
        f"{label}: "
        f"Threshold = {best_threshold:.2f}, "
        f"Validation F1 = {best_f1:.4f}"
    )

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done   4 out of   6 | elapsed:   24.8s remaining:   12.4s
[Parallel(n_jobs=-1)]: Done   6 out of   6 | elapsed:   27.0s finished


toxic: Threshold = 0.20, Validation F1 = 0.7683
severe_toxic: Threshold = 0.15, Validation F1 = 0.4558
obscene: Threshold = 0.20, Validation F1 = 0.7986
threat: Threshold = -0.05, Validation F1 = 0.4294
insult: Threshold = -0.00, Validation F1 = 0.6883
identity_hate: Threshold = 0.05, Validation F1 = 0.4244


In [57]:
test_scores = best_svc.decision_function(X_test)

test_pred_svc = np.zeros_like(test_scores, dtype=int)

for i, threshold in enumerate(best_thresholds_svc):
    test_pred_svc[:, i] = (
        test_scores[:, i] >= threshold
    ).astype(int)

In [58]:
print("Linear SVC - Per Label Threshold Tuned")

print(
    "Macro F1:",
    f1_score(
        y_test,
        test_pred_svc,
        average="macro"
    )
)

print(
    "Micro F1:",
    f1_score(
        y_test,
        test_pred_svc,
        average="micro"
    )
)

print(
    "Hamming Loss:",
    hamming_loss(
        y_test,
        test_pred_svc
    )
)

Linear SVC - Per Label Threshold Tuned
Macro F1: 0.5883296911665917
Micro F1: 0.7154713768886998
Hamming Loss: 0.02094626351245496


In [59]:
best_lr = copy.deepcopy(grid_lr.best_estimator_)

best_lr.fit(X_train_sub, y_train_sub)

val_probs = best_lr.predict_proba(X_val)

best_thresholds_lr = []

for i, label in enumerate(y_train.columns):

    best_threshold = 0.5
    best_f1 = 0

    for threshold in np.arange(0.10, 0.91, 0.05):

        val_pred = (
            val_probs[:, i] >= threshold
        ).astype(int)

        f1 = f1_score(
            y_val.iloc[:, i],
            val_pred
        )

        if f1 > best_f1:
            best_f1 = f1
            best_threshold = threshold

    best_thresholds_lr.append(best_threshold)

    print(
        f"{label}: "
        f"Threshold = {best_threshold:.2f}, "
        f"Validation F1 = {best_f1:.4f}"
    )

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done   4 out of   6 | elapsed:   14.0s remaining:    6.9s
[Parallel(n_jobs=-1)]: Done   6 out of   6 | elapsed:   15.4s finished


toxic: Threshold = 0.60, Validation F1 = 0.7747
severe_toxic: Threshold = 0.80, Validation F1 = 0.5039
obscene: Threshold = 0.70, Validation F1 = 0.8087
threat: Threshold = 0.85, Validation F1 = 0.4103
insult: Threshold = 0.65, Validation F1 = 0.7112
identity_hate: Threshold = 0.80, Validation F1 = 0.4505


In [60]:
test_probs_lr = best_lr.predict_proba(X_test)

test_pred_lr = np.zeros_like(
    test_probs_lr,
    dtype=int
)

for i, threshold in enumerate(best_thresholds_lr):
    test_pred_lr[:, i] = (
        test_probs_lr[:, i] >= threshold
    ).astype(int)

In [61]:
print("Logistic Regression - Per Label Threshold Tuned")

print(
    "Macro F1:",
    f1_score(
        y_test,
        test_pred_lr,
        average="macro"
    )
)

print(
    "Micro F1:",
    f1_score(
        y_test,
        test_pred_lr,
        average="micro"
    )
)

print(
    "Hamming Loss:",
    hamming_loss(
        y_test,
        test_pred_lr
    )
)

Logistic Regression - Per Label Threshold Tuned
Macro F1: 0.6109730021664789
Micro F1: 0.7312140729967449
Hamming Loss: 0.02026737688652149


In [65]:
import joblib

# Get the best estimator/hyperparameters
final_lr = copy.deepcopy(grid_lr.best_estimator_)

# Package final model + thresholds
lr_model_package = {
    "model": final_lr,
    "thresholds": best_thresholds_lr,
    "labels": list(y_train.columns)
}

joblib.dump(
    lr_model_package,
    "best_lr_toxic_comment_model.pkl"
)

['best_lr_toxic_comment_model.pkl']

In [76]:
import joblib
import numpy as np
import pandas as pd


# =========================================================
# 1. LOAD SAVED MODEL
# =========================================================

package = joblib.load("best_lr_toxic_comment_model.pkl")

model = package["model"]
thresholds = package["thresholds"]
labels = package["labels"]

print("Model loaded successfully!")
print("Labels:", labels)
print("Thresholds:", thresholds)


# =========================================================
# 2. TEST COMMENTS
# =========================================================

test_comments = [

    # 🟢 EASY
    "You are an idiot.",
    "I hate you.",
    "You are stupid and worthless.",
    "Shut up, you moron.",
    "You are a disgusting person.",
    "You are a fucking idiot.",
    "I hope you die.",

    # 🟡 EASY → MEDIUM
    "Nobody wants someone like you here.",
    "That was a really stupid thing to say.",
    "You clearly have no idea what you're talking about.",
    "Your comments are completely pointless.",
    "You should be ashamed of yourself.",
    "This discussion would be better without you.",

    # 🟠 MEDIUM
    "That's the dumbest argument I've ever heard.",
    "Your behavior here is absolutely disgusting.",
    "Stop making these ridiculous comments.",
    "I will hurt you if you come here again.",
    "I'm angry about what happened, but I'm not going to threaten anyone.",
    "I strongly disagree with your opinion.",
    "I think this conversation has become unnecessarily hostile."
]


# =========================================================
# 3. TEST EACH COMMENT
# =========================================================

for number, comment in enumerate(test_comments, start=1):

    # -----------------------------------------------------
    # Preprocess comment
    # -----------------------------------------------------

    cleaned_comment = clean_text(comment)


    # -----------------------------------------------------
    # Create DataFrame
    # -----------------------------------------------------

    new_data = pd.DataFrame({
        "clean_comment": [cleaned_comment],
        "comment_length": [len(comment)],
        "word_count": [len(comment.split())]
    })


    # -----------------------------------------------------
    # Get probabilities
    # -----------------------------------------------------

    probs = model.predict_proba(new_data)


    # -----------------------------------------------------
    # Apply per-label thresholds
    # -----------------------------------------------------

    predictions = np.zeros_like(probs, dtype=int)

    for i, threshold in enumerate(thresholds):

        predictions[:, i] = (
            probs[:, i] >= threshold
        ).astype(int)


    # =====================================================
    # DISPLAY RESULT
    # =====================================================

    print("\n" + "=" * 80)
    print(f"COMMENT {number}: {comment}")
    print("=" * 80)

    for i, label in enumerate(labels):

        print(
            f"{label:15} "
            f"Probability = {probs[0, i]:.4f}   "
            f"Threshold = {thresholds[i]:.2f}   "
            f"Prediction = {predictions[0, i]}"
        )

Model loaded successfully!
Labels: ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']
Thresholds: [np.float64(0.6000000000000002), np.float64(0.8000000000000002), np.float64(0.7000000000000002), np.float64(0.8500000000000002), np.float64(0.6500000000000001), np.float64(0.8000000000000002)]

COMMENT 1: You are an idiot.
toxic           Probability = 1.0000   Threshold = 0.60   Prediction = 1
severe_toxic    Probability = 0.7830   Threshold = 0.80   Prediction = 0
obscene         Probability = 0.9998   Threshold = 0.70   Prediction = 1
threat          Probability = 0.0129   Threshold = 0.85   Prediction = 0
insult          Probability = 1.0000   Threshold = 0.65   Prediction = 1
identity_hate   Probability = 0.1669   Threshold = 0.80   Prediction = 0

COMMENT 2: I hate you.
toxic           Probability = 0.9995   Threshold = 0.60   Prediction = 1
severe_toxic    Probability = 0.7478   Threshold = 0.80   Prediction = 0
obscene         Probability = 0.1556   Threshold